In [ ]:
import pandas as pd
from dateutil import parser
import pytz

In [2]:
cfbd = pd.read_csv("/Users/emil/Documents/GitHub/IS477/data/raw/cfbd_merged.csv")
box = pd.read_csv("/Users/emil/Documents/GitHub/IS477/data/raw/cfb_box-scores_2002-2024.csv")

def clean_team_name(name):
    if pd.isna(name):
        return None
    name = str(name).lower().strip()

    replacements = {
        "&": "and",
        "st.": "state",
        "st": "state ",
        "university": "",
        "univ": "",
        "u ": "",
        "-": " ",
        ".":""
    }

    for old, new in replacements.items():
        name = name.replace(old, new)
    
    return " ".join(name.split())

cfbd["home_clean"] = cfbd["homeTeam"].map(clean_team_name)
cfbd["away_clean"] = cfbd["awayTeam"].map(clean_team_name)

box["home_clean"] = box["home"].map(clean_team_name)
box["away_clean"] = box["away"].map(clean_team_name)


def parse_et_to_utc(dt_string):
    if pd.isna(dt_string):
        return None
    local = parser.parse(dt_string)
    eastern = pytz.timezone("US/Eastern")
    local = eastern.localize(local)
    return local.astimezone(pytz.utc)

cfbd["game_datetime_utc"] = pd.to_datetime(cfbd["startDate"], utc=True)

box["game_datetime_utc"] = box["date"].map(parse_et_to_utc)

cfbd["game_datetime_round"] = cfbd["game_datetime_utc"].dt.round("10min")
box["game_datetime_round"] = box["game_datetime_utc"].dt.round("10min")

merged = cfbd.merge(
    box,
    how="inner",
    left_on=["game_datetime_round", "home_clean", "away_clean"],
    right_on=["game_datetime_round", "home_clean", "away_clean"],
    suffixes=("_cfbd","_box")
)

print("Merged sample:")
display(merged.head())

print(f"Merged rows: {len(merged)}")

/var/folders/5l/415lzmwx6x99xzsk4hxsyy980000gp/T/ipykernel_36440/1354495461.py:1: DtypeWarning: Columns (31,40) have mixed types. Specify dtype option on import or set low_memory=False.
  cfbd = pd.read_csv("/Users/emil/Documents/GitHub/IS477/data/raw/cfbd_merged.csv")


Merged sample:


,id_x,season_cfbd,week_cfbd,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,attendance_cfbd,...,int_home,pen_num_away,pen_yards_away,pen_num_home,pen_yards_home,possession_away,possession_home,attendance_box,tv,game_datetime_utc_box
0,401643703,2024,1,regular,2024-08-31T04:00:00.000Z,True,True,False,False,17037.0,...,3.0,13.0,91.0,10.0,105.0,34.37,25.63,17037.0,MWN,2024-08-31 04:00:00+00:00


Merged rows: 1


In [4]:
len(merged)

1